# Selección del modelo

In [14]:
# Cargando las librerias necesarias
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import RFE

# Customizaciones
pd.set_option('display.max_columns', None)

In [5]:
# Cargando los datos de la fase anterior de limpieza desde un fichero pickle
df = pd.read_pickle('./data/df_trans.pkl')

# Verificar el dataframe
print(df.head())
print(df.info())
print(df.shape)

   Diabetes_binary  HighBP  HighChol       BMI  Smoker  Stroke  \
0              0.0     1.0       0.0 -0.602885     0.0     0.0   
1              0.0     1.0       1.0 -0.602885     1.0     1.0   
2              0.0     0.0       0.0 -0.602885     0.0     0.0   
3              0.0     1.0       1.0 -0.281138     1.0     0.0   
4              0.0     0.0       0.0 -0.120264     1.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  Veggies  NoDocbcCost   GenHlth  \
0                   0.0           1.0     0.0      1.0          0.0 -0.131141   
1                   0.0           0.0     1.0      0.0          0.0 -0.131141   
2                   0.0           1.0     1.0      1.0          0.0  1.681222   
3                   0.0           1.0     1.0      1.0          0.0 -0.131141   
4                   0.0           1.0     1.0      1.0          0.0  0.775041   

   MentHlth  PhysHlth  DiffWalk  Sex       Age  Education    Income  
0  0.148542  2.398501       0.0  1.0 -1.618374

## Hacer una selección de variables para reducir el input de los usuarios

In [15]:
X = df.drop(columns=["Diabetes_binary"])
y = df["Diabetes_binary"].astype(int)

# 1. Random Forest Feature Importance
rf = RandomForestClassifier(n_estimators=500, random_state=42)
rf.fit(X, y)
importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

print("=== Importancia de variables (Random Forest) ===")
print(importancias)

# 2. Logistic Regression con L1 (Lasso)
log_l1 = LogisticRegression(penalty="l1", solver="liblinear", random_state=42)
log_l1.fit(X, y)
coefs = pd.Series(np.abs(log_l1.coef_[0]), index=X.columns).sort_values(ascending=False)

print("\n=== Importancia de variables (Logistic L1) ===")
print(coefs)

# 3. Selección de top 10 por RF
top10_rf = importancias.head(10).index.tolist()
# 4. Selección de top 10 por L1
top10_l1 = coefs.head(10).index.tolist()

# Intersección como "más estables"
final_candidates = list(set(top10_rf) & set(top10_l1))

print("\nVariables recomendadas (intersección RF + L1):", final_candidates)

=== Importancia de variables (Random Forest) ===
BMI                     0.171385
Age                     0.130699
GenHlth                 0.104923
Income                  0.086991
HighBP                  0.076202
PhysHlth                0.071780
Education               0.060762
MentHlth                0.054858
HighChol                0.038955
Smoker                  0.030375
Fruits                  0.030187
Sex                     0.027146
DiffWalk                0.025544
PhysActivity            0.024499
Veggies                 0.023323
HeartDiseaseorAttack    0.019061
NoDocbcCost             0.012988
Stroke                  0.010324
dtype: float64

=== Importancia de variables (Logistic L1) ===
HighBP                  0.700934
GenHlth                 0.631376
HighChol                0.581645
BMI                     0.549436
Age                     0.458249
HeartDiseaseorAttack    0.269618
Sex                     0.250962
Stroke                  0.173322
Income                  0.1290

## Hacer el split entre train y test

In [16]:
# Separar variables predictoras (X) y objetivo (y)
# Variables seleccionadas
vars_seleccionadas = final_candidates
target = 'Diabetes_binary'

# Definir X e y solo con las variables seleccionadas
X = df[vars_seleccionadas]
y = df[target].astype(int)

# Split en train y test (ej. 80/20, puedes cambiar si lo deseas)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42  # sin estratificación por ahora
)

print("Shape train:", X_train.shape, "Shape test:", X_test.shape)
print("\nDistribución target en train:\n", y_train.value_counts(normalize=True))
print("\nDistribución target en test:\n", y_test.value_counts(normalize=True))

Shape train: (54174, 6) Shape test: (13544, 6)

Distribución target en train:
 Diabetes_binary
1    0.506941
0    0.493059
Name: proportion, dtype: float64

Distribución target en test:
 Diabetes_binary
1    0.512847
0    0.487153
Name: proportion, dtype: float64


## Preparar un pipeline para entrenar los modelos candidatos

In [17]:
# Definir pipelines de los modelos

pipelines = {
    "LogReg": Pipeline(steps=[
        ("model", LogisticRegression(max_iter=200, n_jobs=None, class_weight=None))
    ]),
    "SVM-RBF": Pipeline(steps=[
        ("model", SVC(kernel="rbf", probability=True))  # datos ya escalados
    ]),
    "RF": Pipeline(steps=[
        ("model", RandomForestClassifier(n_estimators=300, random_state=42))
    ]),
    "GB": Pipeline(steps=[
        ("model", GradientBoostingClassifier(random_state=42))
    ]),
}

## Crear la función para evaluar los modelos

In [18]:
# Función para evaluar modelos

def evaluar_modelo(nombre, pipe, Xtr, ytr, Xte, yte):
    pipe.fit(Xtr, ytr)
    y_pred = pipe.predict(Xte)
    # Para AUC con modelos que soportan probas; si no, usar decision_function
    if hasattr(pipe.named_steps["model"], "predict_proba"):
        y_proba = pipe.predict_proba(Xte)[:, 1]
    else:
        # SVC con probability=True sí tiene predict_proba; de lo contrario usa decision_function
        y_scores = pipe.decision_function(Xte)
        # Escalar scores a [0,1] de forma monotónica no cambia AUC, pero usa directamente scores:
        y_proba = y_scores  # roc_auc_score acepta scores
    acc = accuracy_score(yte, y_pred)
    f1  = f1_score(yte, y_pred)
    auc = roc_auc_score(yte, y_proba)
    print(f"\n=== {nombre} ===")
    print(f"Accuracy: {acc:.4f} | F1: {f1:.4f} | ROC-AUC: {auc:.4f}")
    print(classification_report(yte, y_pred, digits=4))

## Entrenar y evaluar los modelos

In [19]:
# Entrenando y evaluando los modelos

for nombre, pipe in pipelines.items():
    evaluar_modelo(nombre, pipe, X_train, y_train, X_test, y_test)


=== LogReg ===
Accuracy: 0.7397 | F1: 0.7497 | ROC-AUC: 0.8159
              precision    recall  f1-score   support

           0     0.7397    0.7186    0.7290      6598
           1     0.7397    0.7599    0.7497      6946

    accuracy                         0.7397     13544
   macro avg     0.7397    0.7392    0.7393     13544
weighted avg     0.7397    0.7397    0.7396     13544


=== SVM-RBF ===
Accuracy: 0.7448 | F1: 0.7627 | ROC-AUC: 0.8020
              precision    recall  f1-score   support

           0     0.7650    0.6873    0.7241      6598
           1     0.7291    0.7995    0.7627      6946

    accuracy                         0.7448     13544
   macro avg     0.7471    0.7434    0.7434     13544
weighted avg     0.7466    0.7448    0.7439     13544


=== RF ===
Accuracy: 0.6978 | F1: 0.7102 | ROC-AUC: 0.7613
              precision    recall  f1-score   support

           0     0.6967    0.6723    0.6843      6598
           1     0.6988    0.7220    0.7102     